In [2]:
%load_ext autoreload
%autoreload 2

from datetime import datetime, timedelta, date, timezone
from functools import partial
import numpy as np
import polars as pl

from okx.store import OrderbookStore
from okx.recipes.forwards import assign_forwards, build_forwards_pchip, build_forwards_kalman
from okx.recipes.helpers import finalize_binning

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [3]:
store = OrderbookStore(
    data_root="data/okx",
    manifest_path="data/okx/manifest.sqlite"
)

In [4]:
store.clear_cache()

Cleared all caches


In [19]:
def inspect_options(df):
    # Check if expiry is before timeMs
    print(f"Number of rows: {df.height}")
    cols = df.collect_schema().names()
    print(f"Columns:")
    for col in cols:
        print(f"{col:<12}: {str(df.select(col).schema[col]):<7} with {df.select(col).unique().height:<10} unique values and {df.select(col).null_count().item():<5} nulls")

    if 'expiry' in cols and 'T' in cols:
        print("=== Expiry/Tenor checks ===")
        expiry_check = df.filter(pl.col('expiry') < pl.col('timeMs'))
        if expiry_check.height > 0:
            print(f"WARNING: {expiry_check.height} rows where expiry < timeMs")
            print(f" - {expiry_check.head(10)}")
        min_T = df.select('T').min().item()
        print(f"Minimum T excluding swap rows: {min_T} = {min_T * 365 * 24 * 3600} seconds")
        expiry_minus_timeMs = (
            df
            .with_columns([(pl.col('expiry') - pl.col('timeMs')).alias('expiry_minus_timeMs')])
        )
        min_expiry_minus_timeMs = expiry_minus_timeMs.select('expiry_minus_timeMs').min().item()
        print(f"Minimum expiry - timeMs (ms) excluding swap rows: {min_expiry_minus_timeMs} = {min_expiry_minus_timeMs / 1000} seconds")

    # Null and zero checks - prints conditionally

    null_bid = df.filter(pl.col('bid_1_px').is_null())
    null_ask = df.filter(pl.col('ask_1_px').is_null())
    null_both = df.filter(pl.col('bid_1_px').is_null() & pl.col('ask_1_px').is_null())
    if null_bid.height > 0 or null_ask.height > 0 or null_both.height > 0:
        print("=== Null checks ===")
    if null_bid.height > 0:
        print(f"{null_bid.height} rows with null bid")
    if null_ask.height > 0:
        print(f"{null_ask.height} rows with null ask")
    if null_both.height > 0:
        print(f"{null_both.height} rows with null bid and ask")

    zero_bid = df.filter(pl.col('bid_1_px') == 0)
    zero_ask = df.filter(pl.col('ask_1_px') == 0)
    zero_both = df.filter((pl.col('bid_1_px') == 0) & (pl.col('ask_1_px') == 0))
    if zero_bid.height > 0 or zero_ask.height > 0 or zero_both.height > 0:
        print("=== Zero checks ===")
    if zero_bid.height > 0:
        print(f"{zero_bid.height} rows with zero bid")
    if zero_ask.height > 0:
        print(f"{zero_ask.height} rows with zero ask")
    if zero_both.height > 0:
        print(f"{zero_both.height} rows with zero bid and ask")
    print("=== Symbol checks ===")

    symbols = df.select('symbol').unique().to_series().sort().to_list()
    print(f"{len(symbols)} unique symbols")
    print("=== Sorting checks ===")
    
    # Verify df is sorted by [timeMs, T] using Polars
    is_sorted = df.equals(df.sort('timeMs'))
    print(f"DataFrame is sorted by timeMs: {is_sorted}")
    if 'F_bid' in cols:
        print("=== Fitted forwards checks ===")
        agg = (
            df.group_by('timeMs', 'T')
            .agg([
                pl.col("F_bid").unique().count().alias("F_bid_unique_count"),
                pl.col("F_ask").unique().count().alias("F_ask_unique_count"),
            ])
        )
        problem_rows = agg.filter(
            (pl.col("F_bid_unique_count") != 1) | (pl.col("F_ask_unique_count") != 1)
        )
        if problem_rows.height > 0:
            print(f"Found {problem_rows.height} timeMs/T pairs with non-unique forwards")
            print(problem_rows.head())
        else:
            print("All timeMs/T pairs have unique forwards")


In [7]:
start_date = date(2025, 9, 15)
end_date = date(2025, 9, 16)
dates = [start_date + timedelta(days=i) for i in range((end_date - start_date).days)]
print(f"Date range constructed from {dates[0]} to {dates[-1]}")

Date range constructed from 2025-09-15 to 2025-09-15


In [8]:
options_df = store.get(
    inst_type='OPTION',
    inst_family='BTC-USD',
    dates=dates,
    depth=1,
    features=['trim', 'strip', 'nullify', 'drop_nulls', 'tenor', 'parse_option'],
    verbose=True
).sort('timeMs').collect()

[store] Applying transforms for BTC-USD/OPTION (depth=1, binning=None, features=['trim', 'strip', 'nullify', 'drop_nulls', 'tenor', 'parse_option'])
  - applied 'trim'
  - applied 'strip'
  - applied 'nullify'
  - applied 'drop_nulls'
  - applied 'tenor'
  - applied 'parse_option'


In [33]:
del options_df

In [9]:
inspect_options(options_df)

Number of rows: 18907313
Columns:
timeMs      : Int64 with 5267921 unique values and 0 nulls
symbol      : String with 651 unique values and 0 nulls
bid_1_px    : Float64 with 2925 unique values and 51007 nulls
ask_1_px    : Float64 with 2848 unique values and 2315 nulls
expiry      : Int64 with 12 unique values and 0 nulls
T           : Float64 with 11517899 unique values and 0 nulls
strike      : Int64 with 80 unique values and 0 nulls
opt_type    : String with 2 unique values and 0 nulls
=== Expiry/Tenor checks ===
Minimum T excluding swap rows: 0.0009132450532724506 = 28800.096 seconds
Minimum expiry - timeMs (ms) excluding swap rows: 28800096 = 28800.096 seconds
=== Null checks ===
51007 rows with null bid
2315 rows with null ask
=== Symbol checks ===
651 unique symbols
=== Sorting checks ===
DataFrame is sorted by timeMs: True


In [10]:
options_lf = options_df.lazy()
forwards_recipe = build_forwards_kalman
fitted = assign_forwards(store, options_lf, dates, forwards_recipe, 'BTC-USD', binning=None, verbose=True).collect()

Assigning kalman forwards to 1 dates
Time taken to collect data: 0:00:00.000233
Building Kalman-filtered Nelson-Siegel forwards for BTC-USD with provided timestamps
Constructing pillars for BTC-USD with swap and futures using provided timestamps
Time taken to fetch swap and futures: 0:00:02.705715
Time taken to align snapshots: 0:00:00.073320


/Users/pascalandreas/Documents/repositories/final-year-project/okx/recipes/pillars.py:140: UserWarning: Sortedness of columns cannot be checked when 'by' groups provided
  min_futures_time = lf_futures.select(pl.col('timeMs').min()).collect().item()


Time taken to concatenate and filter: 0:00:00.882728
Time taken to index and sort pillars: 0:00:00.000148
Total time taken to prepare pillars: 0:00:03.661911


<sys>:0: UserWarning: Sortedness of columns cannot be checked when 'by' groups provided


Time taken to prepare and collect pillars: 0:00:07.213497
Time taken to build 5267921 snapshots: 0:00:36.526223 


Kalman filter:   0%|          | 0/5267921 [00:00<?, ?it/s]

Time taken to run Kalman filter: 0:08:28.292743
Time taken to convert states to Polars: 0:00:01.111390
Total time taken to build Kalman forwards: 0:09:13.143853
Time taken to fetch forwards: 0:09:14.010862
Time taken to build forward lookup: 0:00:22.800782


Matching forwards: 100%|██████████| 5267921/5267921 [01:24<00:00, 62542.44it/s]


Time taken to match forwards: 0:01:24.935931Matched 18,907,313 / 18,907,313 rows)


In [20]:
inspect_options(fitted)

Number of rows: 18907313
Columns:
timeMs      : Int64   with 5267921    unique values and 0     nulls
symbol      : String  with 651        unique values and 0     nulls
bid_1_px    : Float64 with 2925       unique values and 51007 nulls
ask_1_px    : Float64 with 2848       unique values and 2315  nulls
expiry      : Int64   with 12         unique values and 0     nulls
T           : Float64 with 11517899   unique values and 0     nulls
strike      : Int64   with 80         unique values and 0     nulls
opt_type    : String  with 2          unique values and 0     nulls
F_bid       : Float64 with 11517888   unique values and 0     nulls
F_ask       : Float64 with 11517885   unique values and 0     nulls
=== Expiry/Tenor checks ===
Minimum T excluding swap rows: 0.0009132450532724506 = 28800.096 seconds
Minimum expiry - timeMs (ms) excluding swap rows: 28800096 = 28800.096 seconds
=== Null checks ===
51007 rows with null bid
2315 rows with null ask
=== Symbol checks ===
651 unique symb

In [ ]:
feat_names = ['a', 'b', 'c']
names = ', '.join(feat_names) if len(feat_names) > 1 else feat_names[0]
print(names)